# Глава 15. Масштабирование и производительность

## 15.1. Оптимизация производительности RAG

**Зачем всё это нужно**

Для корпоративных RAG‑систем важны скорость ответа, пропускная способность и стоимость запросов. Они напрямую влияют на удобство пользователя и расходы на эксплуатацию.

**Что оптимизировать в первую очередь**

- Скорость токенизации.
- Эффективность энкодеров.
- Стратегию поиска по векторной базе.
- Интеграцию с языковой моделью (LLM).

**Векторное хранилище**

Выбор и настройка векторного хранилища критически важны. Решения вроде HNSW, FAISS (особенно GPU‑версия), DiskANN или Milvus ускоряют поиск по большому индексу в разы. На практике задержка поиска падает с секунд до миллисекунд даже при миллионах документов.

**Индексация и подготовка данных**

Их нужно делать с расчётом на быстрый поиск:
- подбирать размер сегментов и пересечения;
- объединять тематически связанные фрагменты;
- удалять дубликаты чанков.

Это уменьшает объём бесполезного контекста и снижает нагрузку на генерацию.

**Сжатие и квантизация эмбеддингов**

Это эффективный способ сэкономить память и ускорить поиск. Форматы float8, int8 или эмбеддинги, сжатые через PCA, дают ускорение и снижение стоимости хранения при потере качества меньше 0,5%. В крупных пайплайнах это лучше сочетать с уменьшением размера индекса — точность почти не падает.

## 15.2. Горизонтальное и вертикальное масштабирования

**Масштабирование RAG-систем** — это переход от прототипа к рабочей системе под большой нагрузкой. Есть два основных подхода:

**1. Вертикальное масштабирование** — усиление одного сервера: больше ядер, памяти, быстрые диски, GPU.  
Плюсы: просто управлять, не нужно менять архитектуру.  
Минус: упирается в предел одного сервера.

**2. Горизонтальное масштабирование** — добавление новых серверов в кластер.  
Позволяет распределить нагрузку между:
- репликами векторной базы,
- балансировщиками,
- сервисами эмбеддингов,
- инстансами языковых моделей.

**Как это работает:**

- Векторное хранилище делится на **шарды** (части), часто автоматически.
- Запросы идут к наименее загруженным узлам.
- Если один узел падает, система продолжает работать.
- Данные можно размещать в разных регионах — это снижает задержки и повышает надёжность.

**Автомасштабирование** — система сама добавляет или убирает ресурсы по метрикам: CPU, память, длина очередей.

**Сегментирование данных** даёт почти неограниченную масштабируемость: запросы идут только к нужным шардам, результаты собираются и передаются в языковую модель.

**Лучший вариант — гибрид:**  

- критичные компоненты масштабируются вертикально,  
- легко распараллеливаемые сервисы — горизонтально.

**Сложный случай:** графовые структуры хуже масштабируются горизонтально, потому что нужно обходить связи между узлами. Тут помогают гибридные подходы — графы + векторы.

## 15.3. Кеширование и оптимизация запросов

**Кеширование** — это сохранение уже готовых ответов, чтобы не делать одну и ту же работу дважды. Это сильно ускоряет систему и снижает затраты.

**Основные идеи:**

1. **Кеш в памяти** — хранит популярные запросы и ответы. Время ответа падает с секунд до миллисекунд. Старые и редкие записи удаляются, чтобы не забивать память.

2. **Семантическое кеширование** — ищет не точное совпадение, а похожий по смыслу запрос. Если смысл близкий — возвращает готовый ответ. Ускоряет примерно в 15 раз.

3. **Кеширование промежуточных результатов** — сохраняет эмбеддинги, найденные документы и контекст, чтобы использовать их повторно.

4. **Кеширование промптов** — модель запоминает уже обработанные части запроса. Экономия до 90% стоимости при работе с большими документами. Важно: постоянный контекст лучше ставить в начало промпта.

5. **Умная маршрутизация** — простые вопросы идут к быстрым моделям, сложные — к мощным. Баланс между качеством и скоростью.

6. **Предварительная загрузка** — система предугадывает следующие запросы пользователя и заранее готовит ответы. Ответ приходит почти мгновенно.

7. **Многоуровневый кеш** — популярное хранится в быстрой памяти, менее популярное — в распределённом кеше, редкое — считается заново.

8. **Обновление кеша** — при изменении данных устаревшие записи удаляются или обновляются, чтобы ответы оставались актуальными.

9. **Мониторинг** — система следит за эффективностью кеша и сама настраивает его параметры.

**Суть:** кеширование экономит время и деньги за счёт повторного использования уже сделанной работы.

## 15.4. Архитектурные решения для высоконагруженных систем

- Для высоконагруженных RAG‑систем нужна архитектура, которая выдерживает **тысячи запросов в секунду**, работает **надёжно** и **масштабируется**.
- Важно сохранять **низкие задержки** и **высокую доступность**.
- Систему лучше делить на **независимые микросервисы**:  
  - модель эмбеддингов,  
  - векторный поиск,  
  - оркестрация,  
  - генерация ответов.
- Каждый сервис можно **масштабировать отдельно**, потому что у них разные требования к ресурсам.
- Слабая связанность упрощает **обновления** и позволяет **заменять части системы без остановки** всей работы.